# Offline pipeline on the new clips, on a Kaggle GPU

Runs `pipeline.py`, the offline analysis, on the videos of the attached dataset:
court, players, player analysis, ball (YOLOv8x on full-resolution tiles, on the
GPU, then the path over the whole clip) and shots. Its results are the reference
to compare the real-time pipeline with (`notebooks/realtime_run_kaggle.ipynb`).

The results come out as **`offline_outputs.zip`**, with the same folders as the
project (`outputs/<clip>/...`): unzip it in the project folder on your PC. The
ball CSVs alone are also in `ball/`.

**Setup.** The same private dataset as the real-time notebook: the videos and
`ball_tracker.pt` (the offline YOLOv8x). *Settings*: GPU, Internet on. Then
*Save Version -> Save & Run All*.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: set Settings -> Accelerator to a GPU'

## 2. Get the code

In [ ]:
BRANCH = 'realtime-motion-ball'
!git clone -q -b {BRANCH} https://github.com/marcomassa-unitn/tennis_players_tracking.git
%cd tennis_players_tracking
!pip -q install ultralytics

## 3. Find the videos and the weights
`FIRST = 12` keeps only the videos named `input12`, `input13`, ... (`None` for all).

In [ ]:
import glob, os, re, shutil
files = glob.glob('/kaggle/input/**/*', recursive=True)
VIDEOS = sorted(f for f in files if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')))
MODEL = next((f for f in files if os.path.basename(f) == 'ball_tracker.pt'), None)
assert MODEL, 'ball_tracker.pt not found: add the offline YOLOv8x weights to the dataset'

FIRST = 12
if FIRST is not None:
    def wanted(v):
        m = re.fullmatch(r'input(\d+)', os.path.splitext(os.path.basename(v))[0])
        return bool(m) and int(m.group(1)) >= FIRST
    VIDEOS = [v for v in VIDEOS if wanted(v)]
assert VIDEOS, 'No video to analyse'
print('Model:', MODEL)
print(len(VIDEOS), 'videos:', *[os.path.basename(v) for v in VIDEOS], sep='\n  ')

## 4. Run the offline pipeline on every clip
Everything but motion estimation (optical flow, block matching), which nothing
else reads. Ball detection runs on the GPU; the rest on the CPU.

In [ ]:
failed = []
for v in VIDEOS:
    stem = os.path.splitext(os.path.basename(v))[0]
    print('=' * 70, '\n', stem)
    !python pipeline.py --video "{v}" --model "{MODEL}" --redo all --skip-motion --skip-evaluation
    if not os.path.exists(f'outputs/{stem}/ball_coordinates/ball_{stem}.csv'):
        failed.append(stem)
print('Failed:', failed or 'none')

## 5. Summary

In [ ]:
import pandas as pd
rows = []
for v in VIDEOS:
    stem = os.path.splitext(os.path.basename(v))[0]
    ball = f'outputs/{stem}/ball_coordinates/ball_{stem}.csv'
    shots = f'outputs/{stem}/shot_analysis/shots.csv'
    if not os.path.exists(ball):
        rows.append({'clip': stem, 'ball positions': None})
        continue
    b = pd.read_csv(ball)
    rows.append({'clip': stem, 'ball positions': int((b.interpolated == 0).sum()),
                 'shots': len(pd.read_csv(shots)) if os.path.exists(shots) else None})
pd.DataFrame(rows)

## 6. Save the results
Only `/kaggle/working` is kept. Download `offline_outputs.zip`.

In [ ]:
out = '/kaggle/working'
os.makedirs(f'{out}/ball', exist_ok=True)
for v in VIDEOS:
    stem = os.path.splitext(os.path.basename(v))[0]
    p = f'outputs/{stem}/ball_coordinates/ball_{stem}.csv'
    if os.path.exists(p):
        shutil.copy(p, f'{out}/ball/')
stems = [os.path.splitext(os.path.basename(v))[0] for v in VIDEOS]
!cd /kaggle/working/tennis_players_tracking && zip -q -r /kaggle/working/offline_outputs.zip {' '.join(f'"outputs/{s}"' for s in stems if os.path.isdir(f'outputs/{s}'))}
print(sorted(os.listdir(out)))
%cd /kaggle/working
!rm -rf tennis_players_tracking